# Interactive Decision Tree - Oracle ve Sample Data Demo

Bu notebook iki akisi gosterir:

1. Notebook icinde uretilen sample pandas `DataFrame` ile UI'i acmak.
2. Oracle'a farkli bir demo dataset yazmak, Oracle'dan geri okumak ve okunan datayi UI'da kullanmak.

Credential bilgileri `oracle_config/ora_config.ini` icinden okunur; kullanici/sifre ekrana yazdirilmaz ve `oracle_config/` git'e dahil edilmez.


## 0. Kurulum notu

Bu notebook'u repo kokunden calistiriyorsan once bir kez sunu calistir:

```powershell
.\.venv\Scripts\python.exe -m pip install -e ".[notebook,oracle]"
```

Notebook kernel'in bu `.venv` degilse, asagidaki `%pip install -e ".[notebook,oracle]"` satirini acip bir kez calistirabilirsin.


In [ ]:
# Import hatasi alirsan once kernel'in `interactive_decision_tree_env (.venv)` oldugunu kontrol et.
# Gerekirse bu satiri acip bir kez calistir:
# %pip install -e ".[notebook,oracle]"

import sys
from configparser import ConfigParser
from pathlib import Path
from urllib.parse import quote_plus

IGNORED_ROOT_PARTS = {".trash", "trash", ".trash-1000", ".trash-1001", "$recycle.bin"}


def is_ignored_project_candidate(path: Path) -> bool:
    return any(part.lower() in IGNORED_ROOT_PARTS for part in path.parts)


def looks_like_project_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "interactive_decision_tree").is_dir()


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []
    for path in (cwd, *cwd.parents):
        candidates.append(path)
        candidates.append(path / "interactive_decision_tree")
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if is_ignored_project_candidate(candidate):
            continue
        if looks_like_project_root(candidate):
            return candidate
    return cwd


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("python_executable:", sys.executable)
print("project_root:", PROJECT_ROOT)

try:
    import numpy as np
    import pandas as pd
    from sqlalchemy import create_engine, text
    from interactive_decision_tree import launch_tree, launch_tree_sql
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Notebook kernel'inde eksik paket var veya yanlis kernel secili. "
        "VS Code/Jupyter kernel olarak `interactive_decision_tree_env (.venv)` sec. "
        f"Aktif Python: {sys.executable}. Eksik modul: {exc.name}. "
        'Gerekirse bu hucreden once `%pip install -e \".[notebook,oracle]\"` calistir.'
    ) from exc


## 0.1 UI link ayarlari

Lokal makinede default ayarlar yeterli. OpenShift/Jupyter proxy veya route kullaniyorsan `APP_BASE_URL`, `APP_HOST` ve `APP_SCHEME` alanlarini bu hucrede degistir.


In [ ]:
import inspect
from urllib.parse import urlsplit, urlunsplit

APP_PORT = 8501
APP_START_SERVER = True
APP_OPEN_BROWSER = True

# OpenShift/Jupyter proxy icin ornek:
# APP_START_SERVER = False
# APP_OPEN_BROWSER = False
# APP_BASE_URL = "https://<notebook-host>/notebook/<workspace>/proxy/8501/"
APP_BASE_URL = ""

# Route veya farkli host icin ornek:
# APP_HOST = "interactive-tree.apps.internal"
# APP_SCHEME = "https"
APP_HOST = "localhost"
APP_SCHEME = "http"


def callable_accepts_kwargs(func) -> bool:
    signature = inspect.signature(func)
    return any(param.kind == inspect.Parameter.VAR_KEYWORD for param in signature.parameters.values())


def filter_supported_kwargs(func, kwargs: dict) -> dict:
    if callable_accepts_kwargs(func):
        return kwargs
    supported = inspect.signature(func).parameters
    return {key: value for key, value in kwargs.items() if key in supported}


def ui_launch_kwargs(func=launch_tree) -> dict:
    # Sadece server davranisi argumanlarini geciyoruz. URL host/proxy donusumu
    # format_ui_url() ile yapiliyor; boylece eski launch_tree surumlerinde de
    # `unexpected keyword argument host` hatasi alinmiyor.
    return filter_supported_kwargs(
        func,
        {
            "port": APP_PORT,
            "start_server": APP_START_SERVER,
            "open_browser": APP_OPEN_BROWSER,
        },
    )


def append_query(base_url: str, query: str) -> str:
    parts = urlsplit(base_url)
    path = parts.path or "/"
    merged_query = f"{parts.query}&{query}" if parts.query else query
    return urlunsplit((parts.scheme, parts.netloc, path, merged_query, parts.fragment))


def format_ui_url(url: str) -> str:
    parsed = urlsplit(url)
    query = parsed.query
    if APP_BASE_URL:
        return append_query(APP_BASE_URL, query)
    if APP_HOST != "localhost" or APP_SCHEME != "http":
        host = APP_HOST if "://" not in APP_HOST else urlsplit(APP_HOST).netloc
        netloc = host if ":" in host else f"{host}:{APP_PORT}"
        base_url = f"{APP_SCHEME}://{netloc}/"
        return append_query(base_url, query)
    return url


ui_launch_kwargs(), format_ui_url(f"http://localhost:{APP_PORT}/?data_id=demo&work_id=demo")


## 1. Notebook icinde sample DataFrame uretme

Bu bolum dis kaynaga baglanmadan RAM'de sample data uretir ve UI'a aktarir.


In [ ]:
rng = np.random.default_rng(7)
n = 500

sample_df = pd.DataFrame(
    {
        "age": rng.integers(21, 72, size=n),
        "income": rng.normal(52_000, 18_000, size=n).clip(12_000, 140_000).round(2),
        "tenure_months": rng.integers(0, 120, size=n),
        "segment": rng.choice(["A", "B", "C", "D", "E"], size=n),
        "channel": rng.choice(["branch", "mobile", "web", "call_center"], size=n),
        "region": rng.choice(["marmara", "ege", "akdeniz", "ic_anadolu"], size=n),
    }
)

sample_score = (
    (sample_df["income"] < 42_000).astype(int)
    + (sample_df["tenure_months"] < 18).astype(int)
    + sample_df["segment"].isin(["C", "D"]).astype(int)
    + (sample_df["channel"] == "mobile").astype(int)
    - (sample_df["age"] > 55).astype(int)
)
sample_df["risk_flag"] = np.where(sample_score >= 2, "high_risk", "low_risk")

sample_df.head()


In [ ]:
sample_url = format_ui_url(
    launch_tree(
        sample_df,
        target="risk_flag",
        features=["age", "income", "tenure_months", "segment", "channel", "region"],
        session_name="Notebook sample data",
        **ui_launch_kwargs(launch_tree),
    )
)
sample_url


## 2. Oracle baglantisini acma

Bu hucre `oracle_config/ora_config.ini` veya `ora_config/ora_config.ini` dosyasini okur, SQLAlchemy Oracle URL'ini bellekte olusturur ve `select 1 from dual` ile baglantiyi test eder.


In [ ]:
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
    if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

oracle_config_candidates = [
    PROJECT_ROOT / "oracle_config" / "ora_config.ini",
    PROJECT_ROOT / "ora_config" / "ora_config.ini",
    Path.cwd().resolve() / "oracle_config" / "ora_config.ini",
    Path.cwd().resolve() / "ora_config" / "ora_config.ini",
    Path.cwd().resolve().parent / "oracle_config" / "ora_config.ini",
    Path.cwd().resolve().parent / "ora_config" / "ora_config.ini",
]

oracle_config_path = next((path for path in oracle_config_candidates if path.exists()), None)
if oracle_config_path is None:
    searched_paths = "\n".join(str(path) for path in oracle_config_candidates)
    raise FileNotFoundError(
        "oracle_config/ora_config.ini bulunamadi. Aranan yollar:\n" + searched_paths
    )

oracle_section = "ORA_PRD_ZTUSER"
parser = ConfigParser()
parser.read(oracle_config_path, encoding="utf-8")
if oracle_section not in parser:
    raise KeyError(f"INI icinde bolum bulunamadi: {oracle_section}")

cfg = parser[oracle_section]
oracle_url = (
    "oracle+oracledb://"
    f"{quote_plus(cfg['user'])}:{quote_plus(cfg['password'])}"
    f"@{cfg['host']}:{cfg.get('port', '1521')}/?service_name={quote_plus(cfg['service_name'])}"
)

engine = create_engine(oracle_url)
try:
    with engine.connect() as conn:
        ok_value = conn.execute(text("select 1 as ok from dual")).scalar_one()
finally:
    engine.dispose()

print({
    "section": oracle_section,
    "config_path": str(oracle_config_path),
    "connection": "ok",
    "select_1": int(ok_value),
})



## 3. Oracle icin farkli demo data uretme

Bu dataset lokal sample datadan farklidir. Sonraki hucrede Oracle'a tablo olarak yazilir.


In [ ]:
oracle_rng = np.random.default_rng(42)
oracle_n = 320

oracle_demo_df = pd.DataFrame(
    {
        "CUSTOMER_ID": np.arange(1, oracle_n + 1),
        "AGE": oracle_rng.integers(22, 76, size=oracle_n),
        "INCOME": oracle_rng.normal(64_000, 22_000, size=oracle_n).clip(15_000, 180_000).round(2),
        "TENURE_MONTHS": oracle_rng.integers(0, 144, size=oracle_n),
        "SEGMENT": oracle_rng.choice(["SME", "MASS", "AFFLUENT", "YOUNG"], size=oracle_n),
        "CHANNEL": oracle_rng.choice(["BRANCH", "MOBILE", "WEB", "CALL_CENTER"], size=oracle_n),
        "REGION": oracle_rng.choice(["MARMARA", "EGE", "AKDENIZ", "IC_ANADOLU", "KARADENIZ"], size=oracle_n),
        "UTILIZATION": oracle_rng.beta(2.2, 4.5, size=oracle_n).round(4),
    }
)

oracle_score = (
    (oracle_demo_df["INCOME"] < 48_000).astype(int)
    + (oracle_demo_df["TENURE_MONTHS"] < 24).astype(int)
    + oracle_demo_df["SEGMENT"].isin(["SME", "YOUNG"]).astype(int)
    + (oracle_demo_df["CHANNEL"] == "MOBILE").astype(int)
    + (oracle_demo_df["UTILIZATION"] > 0.52).astype(int)
    - (oracle_demo_df["AGE"] > 60).astype(int)
)
oracle_demo_df["RISK_FLAG"] = np.where(oracle_score >= 3, "high_risk", "low_risk")

oracle_demo_df.head()


## 4. Oracle'a yazma

Bu hucre mevcut kullanicinin schema'sinda `IDT_DEMO_TREE_DATA` tablosunu olusturur veya replace eder. Calistirmadan once bu tablo adinin senin ortaminda uygun oldugunu kontrol et.


In [ ]:
import re

from sqlalchemy import Float, Integer, String
from sqlalchemy.exc import SQLAlchemyError

oracle_table_name = "IDT_DEMO_TREE_DATA"
if not re.fullmatch(r"[A-Z][A-Z0-9_]{0,29}", oracle_table_name):
    raise ValueError("Oracle tablo adi buyuk harf, rakam ve underscore icermeli; 30 karakteri asmamali.")

oracle_write_df = oracle_demo_df.copy().where(pd.notna(oracle_demo_df), None)
oracle_sql_dtypes = {
    "CUSTOMER_ID": Integer(),
    "AGE": Integer(),
    "INCOME": Float(precision=53),
    "TENURE_MONTHS": Integer(),
    "SEGMENT": String(30),
    "CHANNEL": String(30),
    "REGION": String(30),
    "UTILIZATION": Float(precision=53),
    "RISK_FLAG": String(30),
}


def oracle_error_text(exc: BaseException) -> str:
    orig = getattr(exc, "orig", "")
    return f"{exc} {orig}".upper()


def oracle_table_exists(conn, table_name: str) -> bool:
    return bool(
        conn.execute(
            text("select count(*) from user_tables where table_name = :table_name"),
            {"table_name": table_name.upper()},
        ).scalar_one()
    )


engine = create_engine(oracle_url)
row_count = None
write_mode = "create"
try:
    with engine.connect() as conn:
        table_exists = oracle_table_exists(conn, oracle_table_name)

    if table_exists:
        try:
            with engine.begin() as conn:
                conn.execute(text(f"drop table {oracle_table_name} purge"))
            table_exists = False
            write_mode = "drop_create"
        except SQLAlchemyError as exc:
            error_text = oracle_error_text(exc)
            if "ORA-00942" in error_text:
                table_exists = False
                write_mode = "create"
            elif "ORA-01031" in error_text:
                with engine.begin() as conn:
                    conn.execute(text(f"delete from {oracle_table_name}"))
                write_mode = "delete_append"
            else:
                raise

    oracle_write_df.to_sql(
        oracle_table_name,
        con=engine,
        if_exists="append",
        index=False,
        chunksize=1000,
        dtype=oracle_sql_dtypes,
    )
    with engine.connect() as conn:
        row_count = conn.execute(text(f"select count(*) from {oracle_table_name}")).scalar_one()
finally:
    engine.dispose()

print({"table": oracle_table_name, "rows_written": int(row_count), "mode": write_mode})


## 5. Oracle'dan okuyup UI'da kullanma

Bu hucre datayi Oracle tablosundan geri cagirir ve `launch_tree_sql(...)` ile UI'a aktarir. UI tarafinda data snapshot olarak `.tree_sessions/` altina kaydedilir; sayfa yenilenince Oracle sorgusu tekrar calismaz.


In [ ]:
oracle_read_query = f"""
select
    CUSTOMER_ID,
    AGE,
    INCOME,
    TENURE_MONTHS,
    SEGMENT,
    CHANNEL,
    REGION,
    UTILIZATION,
    RISK_FLAG
from {oracle_table_name}
"""

oracle_ui_url = format_ui_url(
    launch_tree_sql(
        oracle_url,
        query=oracle_read_query,
        target="RISK_FLAG",
        full_table=True,
        session_name="Oracle roundtrip demo data",
        **ui_launch_kwargs(launch_tree_sql),
    )
)
oracle_ui_url


## 6. Final agaci notebook'a geri yukleme

UI'da agaci finalize ettikten sonra en alttaki `Tree export` bolumunden `Download runnable tree pickle` ile dosyayi indir. Sonra ayni veya baska bir notebook'ta asagidaki gibi acabilirsin.

Not: Pickle dosyalarini sadece kendi urettigin guvenilir dosyalardan yukle.


In [ ]:
from interactive_decision_tree import load_tree_pickle, load_tree_json, score_tree_payload

# UI export pickle dosyasini Downloads klasorune indirdiysen:
tree_pickle_path = Path.home() / "Downloads" / "interactive_entropy_tree_runnable.pkl"
tree_payload = load_tree_pickle(tree_pickle_path)

print("loaded_tree_path:", tree_pickle_path)
print("loaded_tree_modified:", pd.Timestamp.fromtimestamp(tree_pickle_path.stat().st_mtime))
print("loaded_tree_nodes:", tree_payload.get("node_count"))

# JSON indirdiysen:
# tree_payload = load_tree_json(Path.home() / "Downloads" / "interactive_entropy_tree_runnable.json")

# tree_payload.keys()
# tree_payload["tree"]
# tree_payload["metrics"]


## 7. Tek musteri icin skorlama

Bu hucrede musteri degiskenlerini notebook icinde yaratip yukledigimiz gercek pickle/JSON payload ile skorlariz. `tree_payload` bir onceki hucrede UI exportundan yuklenmis olmalidir.

`prediction` global bir threshold'a gore degil, musteri hangi leaf'e dustuyse o leaf'in cogunluk sinifina gore gelir. Binary target icin `positive_class_probability` leaf icindeki positive class oranidir; `prediction_probability` tahmin edilen sinifin leaf icindeki oranidir.


In [ ]:
if "tree_payload" not in globals():
    raise RuntimeError("Once UI export pickle/JSON dosyasini tree_payload degiskenine yukle.")

payload_features = {str(feature) for feature in tree_payload.get("features", [])}

if {"AGE", "INCOME", "TENURE_MONTHS", "SEGMENT", "CHANNEL", "REGION", "UTILIZATION"}.issubset(payload_features):
    one_customer = {
        "CUSTOMER_ID": 999001,
        "AGE": 34,
        "INCOME": 35_000,
        "TENURE_MONTHS": 12,
        "SEGMENT": "SME",
        "CHANNEL": "MOBILE",
        "REGION": "MARMARA",
        "UTILIZATION": 0.62,
    }
else:
    one_customer = {
        "age": 34,
        "income": 35_000,
        "tenure_months": 12,
        "segment": "C",
        "channel": "mobile",
        "region": "marmara",
    }

score_result = score_tree_payload(tree_payload, one_customer)

print("prediction:", score_result["prediction"])
print("prediction_proba:", score_result["prediction_probability"])
print("positive_class:", score_result["positive_class"])
print("positive_class_proba:", score_result["positive_class_probability"])
print("leaf_node_id:", score_result["leaf_node_id"])
print("leaf_path:", score_result["leaf_path"])
if score_result.get("exported_leaf_path") != score_result.get("leaf_path"):
    print("exported_leaf_path:", score_result.get("exported_leaf_path"))
display(pd.DataFrame([score_result["class_probabilities"]]))
pd.DataFrame(score_result["trace"])
